#### import Required Libraries


In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run "/Workspace/Users/abdulm63633@gmail.com/Ecommerce Lakehouse Project/01_setup_file/setup_utils"

In [0]:
print(bronze_schema,silver_schema,gold_schema)

In [0]:
dbutils.widgets.text("catalog", "ecommerce_lakehouse_project", "Catalog")
dbutils.widgets.text("data_source", "products", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")



In [0]:
df_silver = spark.sql(f"SELECT * FROM {catalog}.{silver_schema}.{data_source};")
display(df_silver.limit(20))

In [0]:
df_gold = df_silver.select("product_id","product_name","category","brand","price","created_at")
df_gold.show()

In [0]:
# ============================================================
# GOLD DIM_CUSTOMERS WRITE LOGIC
# ============================================================

# ============================================================
# FIRST RUN -> CREATE TABLE
# ============================================================



# Defensive deduplication before MERGE
df_gold = df_gold.dropDuplicates(["product_id"])


# ============================================================
# FIRST RUN -> CREATE TABLE
# ============================================================

if not spark.catalog.tableExists(
    f"{catalog}.{gold_schema}.dim_{data_source}"
):

    (
        df_gold.write
            .format("delta")

            # Enable Change Data Feed for downstream incremental tracking
            .option("delta.enableChangeDataFeed", "true")

            # Initial table creation
            .mode("overwrite")

            .saveAsTable(
                f"{catalog}.{gold_schema}.dim_{data_source}"
            )
    )

    print(
        f"Successfully created Gold table: dim_{data_source}"
    )


# ============================================================
# INCREMENTAL RUN -> UPSERT CHANGES
# ============================================================

else:

    print(
        f"Running incremental MERGE for dim_{data_source}"
    )

    delta_table = DeltaTable.forName(
        spark,
        f"{catalog}.{gold_schema}.dim_{data_source}"
    )

    (
        delta_table.alias("target").merge(
            source=df_gold.alias("source"),
            condition="""
                target.product_id = source.product_id
            """
        )

        # Update ONLY if business values changed
        .whenMatchedUpdate(
        condition="""
        NOT (target.product_name <=> source.product_name)
        OR NOT (target.category <=> source.category)
        OR NOT (target.brand <=> source.brand)
        OR NOT (target.price <=> source.price)
        OR NOT (target.created_at <=> source.created_at)
    """,
        set={
             "product_name": "coalesce(source.product_name, target.product_name)",
            "category": "coalesce(source.category, target.category)",
            "brand": "coalesce(source.brand, target.brand)",
            "price": "coalesce(source.price, target.price)",
            "created_at": "coalesce(source.created_at, target.created_at)"
            
        }
        )

        # Insert new customers
        .whenNotMatchedInsert(
        values={
            "product_id": "source.product_id",
            "product_name": "source.product_name",
            "category": "source.category",
            "brand": "source.brand",
            "price": "source.price",
            "created_at":"source.created_at"
        }
        )

        .execute()
    )

    print(
        f"Successfully merged incremental data into dim_{data_source}"
    )





